# LoRA Fine-Tuning: Ministral 3 3B on Medical Flashcards

This notebook demonstrates **LoRA (Low-Rank Adaptation)** fine-tuning of `mistralai/Ministral-3-3B-Instruct-2512` on a medical Q&A dataset using Training Hub.

**What is LoRA?** LoRA adds small, trainable low-rank matrices to the model's attention and MLP layers. Instead of updating all 3 billion parameters, only ~1–2% of parameters are trained — dramatically reducing memory requirements and training time.

**Goal:** Teach the model to produce concise, factual medical answers using parameter-efficient fine-tuning that runs on a single GPU.

**Dataset:** [medalpaca/medical_meadow_medical_flashcards](https://huggingface.co/datasets/medalpaca/medical_meadow_medical_flashcards) — 33,955 medical Q&A pairs covering anatomy, pharmacology, pathology, and clinical medicine.

**Hardware requirements:**
- Minimum: 1× GPU with 24GB+ VRAM (e.g., RTX 4090, A5000)
- With QLoRA (4-bit): 1× GPU with 16GB+ VRAM
- Recommended: 1× A100-80GB for fastest training

**Key LoRA parameters:**
- `lora_r` — Rank of the low-rank matrices (higher = more capacity, more memory)
- `lora_alpha` — Scaling factor (typically 2× the rank)
- `load_in_4bit` — Enable QLoRA for reduced memory

## 1. Setup

In [1]:
%pip install 'training-hub[lora]' --quiet

/mnt/vde/workspace/osilkin/dev-repos/mistral-qwen-notebook/training_hub/.venv/bin/python: No module named pip
Note: you may need to restart the kernel to use updated packages.


In [2]:
import json
import os
from pathlib import Path

from datasets import load_dataset
import torch

if torch.cuda.is_available():
    gpu_name = torch.cuda.get_device_name(0)
    props = torch.cuda.get_device_properties(0)
    # Some versions of PyTorch use 'total_memory' instead of 'total_mem'
    gpu_mem = getattr(props, "total_memory", None)
    if gpu_mem is None:
        # fallback for future/older torch versions, raise or warn
        print("WARNING: Unable to get total GPU memory. PyTorch version may be incompatible.")
        print(f"GPU: {gpu_name}")
    else:
        gpu_mem_gb = gpu_mem / 1e9
        print(f"GPU: {gpu_name} ({gpu_mem_gb:.1f} GB)")
else:
    print("WARNING: No GPU detected. LoRA training requires a CUDA GPU.")

GPU: NVIDIA A100-SXM4-80GB (85.0 GB)


## 2. Load and Explore the Dataset

The [medical_meadow_medical_flashcards](https://huggingface.co/datasets/medalpaca/medical_meadow_medical_flashcards) dataset contains concise clinical Q&A pairs in the style of medical flashcards.

In [3]:
ds = load_dataset("medalpaca/medical_meadow_medical_flashcards", split="train")
print(f"Full dataset: {len(ds):,} samples")
print(f"Columns: {ds.column_names}")

# Show a few samples
for i in range(3):
    print(f"\n--- Sample {i} ---")
    print(f"  Q: {ds[i]['input'][:120]}")
    print(f"  A: {ds[i]['output'][:120]}")

Full dataset: 33,955 samples
Columns: ['input', 'output', 'instruction']

--- Sample 0 ---
  Q: What is the relationship between very low Mg2+ levels, PTH levels, and Ca2+ levels?
  A: Very low Mg2+ levels correspond to low PTH levels which in turn results in low Ca2+ levels.

--- Sample 1 ---
  Q: What leads to genitourinary syndrome of menopause (atrophic vaginitis)?
  A: Low estradiol production leads to genitourinary syndrome of menopause (atrophic vaginitis).

--- Sample 2 ---
  Q: What does low REM sleep latency and experiencing hallucinations/sleep paralysis suggest?
  A: Low REM sleep latency and experiencing hallucinations/sleep paralysis suggests narcolepsy.


## 3. Prepare Training Data

Convert the dataset to JSONL **messages** format (chat template) expected by Training Hub's LoRA backend.

In [4]:
def convert_to_messages(row):
    """Convert a flashcard row to chat messages format."""
    return {
        "messages": [
            {"role": "user", "content": row["input"]},
            {"role": "assistant", "content": row["output"]},
        ]
    }


# Preview the converted format
sample = convert_to_messages(ds[0])
print(json.dumps(sample, indent=2))

{
  "messages": [
    {
      "role": "user",
      "content": "What is the relationship between very low Mg2+ levels, PTH levels, and Ca2+ levels?"
    },
    {
      "role": "assistant",
      "content": "Very low Mg2+ levels correspond to low PTH levels which in turn results in low Ca2+ levels."
    }
  ]
}


In [5]:
# Use a subset for this demo
SUBSET_SIZE = 3000
ds_subset = ds.shuffle(seed=42).select(range(SUBSET_SIZE))

OUTPUT_DIR = "./ministral_lora_output"
DATA_PATH = os.path.join(OUTPUT_DIR, "medical_flashcards.jsonl")
os.makedirs(OUTPUT_DIR, exist_ok=True)

with open(DATA_PATH, "w") as f:
    for row in ds_subset:
        f.write(json.dumps(convert_to_messages(row)) + "\n")

print(f"Wrote {SUBSET_SIZE} samples to {DATA_PATH}")

Wrote 3000 samples to ./ministral_lora_output/medical_flashcards.jsonl


## 4. Configure and Run LoRA Training

**Key settings:**
- `lora_r=32` — Rank 32 gives good capacity for medical knowledge
- `lora_alpha=64` — Alpha = 2× rank is a common default
- `load_in_4bit=False` — Set to `True` for QLoRA if your GPU has limited memory
- `sample_packing=True` — Packs short samples together for better GPU utilization

In [6]:
# ── Model ──
MODEL_NAME = "mistralai/Ministral-3-3B-Instruct-2512"
CKPT_DIR = os.path.join(OUTPUT_DIR, "checkpoints")

# ── LoRA ──
LORA_R = 32
LORA_ALPHA = 64
LORA_DROPOUT = 0.0  # 0.0 is optimized for Unsloth

# ── Training ──
NUM_EPOCHS = 3
LEARNING_RATE = 1e-4
MAX_SEQ_LEN = 2048
EFFECTIVE_BATCH_SIZE = 32
WARMUP_STEPS = 10

# ── Quantization (set True for QLoRA if GPU memory is limited) ──
USE_QLORA = False

quant_mode = "QLoRA (4-bit)" if USE_QLORA else "Full precision"
print(f"Model:          {MODEL_NAME}")
print(f"LoRA rank:      {LORA_R}, alpha: {LORA_ALPHA}")
print(f"Quantization:   {quant_mode}")
print(f"Epochs:         {NUM_EPOCHS}")
print(f"Learning rate:  {LEARNING_RATE}")
print(f"Max seq len:    {MAX_SEQ_LEN}")
print(f"Data:           {DATA_PATH} ({SUBSET_SIZE} samples)")

Model:          mistralai/Ministral-3-3B-Instruct-2512
LoRA rank:      32, alpha: 64
Quantization:   Full precision
Epochs:         3
Learning rate:  0.0001
Max seq len:    2048
Data:           ./ministral_lora_output/medical_flashcards.jsonl (3000 samples)


In [7]:
from training_hub import lora_sft

result = lora_sft(
    model_path=MODEL_NAME,
    data_path=DATA_PATH,
    ckpt_output_dir=CKPT_DIR,
    # LoRA configuration
    lora_r=LORA_R,
    lora_alpha=LORA_ALPHA,
    lora_dropout=LORA_DROPOUT,
    # Training
    num_epochs=NUM_EPOCHS,
    effective_batch_size=EFFECTIVE_BATCH_SIZE,
    learning_rate=LEARNING_RATE,
    max_seq_len=MAX_SEQ_LEN,
    warmup_steps=WARMUP_STEPS,
    # Quantization
    load_in_4bit=USE_QLORA,
    # Optimization
    bf16=True,
    sample_packing=True,
    # Ministral 3 is loaded by Unsloth as a VLM — disable assistant_only_loss
    assistant_only_loss=False,
    # Dataset format
    dataset_type="chat_template",
    field_messages="messages",
    # Logging
    logging_steps=10,
    save_steps=500,
    save_total_limit=3,
)

print("\nLoRA training complete!")

/mnt/vde/workspace/osilkin/dev-repos/mistral-qwen-notebook/training_hub/src/training_hub/algorithms/lora.py:77: UserWarning: WARNING: Unsloth should be imported before [transformers, peft] to ensure all optimizations are applied. Your code may run slower or encounter memory issues without these optimizations.

Please restructure your imports with 'import unsloth' at the top of your file.
  from unsloth import FastLanguageModel


🦥 Unsloth: Will patch your computer to enable 2x faster free finetuning.
🦥 Unsloth Zoo will now patch everything to make training faster!
==((====))==  Unsloth 2026.3.6: Fast Ministral3 patching. Transformers: 5.3.0.
   \\   /|    NVIDIA A100-SXM4-80GB. Num GPUs = 8. Max memory: 79.138 GB. Platform: Linux.
O^O/ \_/ \    Torch: 2.10.0+cu128. CUDA: 8.0. CUDA Toolkit: 12.8. Triton: 3.6.0
\        /    Bfloat16 = TRUE. FA [Xformers = 0.0.35. FA2 = True]
 "-____-"     Free license: http://github.com/unslothai/unsloth
Unsloth: Fast downloading is enabled - ignore downloading bars which are red colored!
Unsloth: QLoRA and full finetuning all not selected. Switching to 16bit LoRA.


Loading weights:   0%|          | 0/458 [00:00<?, ?it/s]

Unsloth: Making `model.base_model.model.model.vision_tower.transformer` require gradients


Generating train split: 0 examples [00:00, ? examples/s]

...

Unsloth: Tokenizing ["text"] (num_proc=64):   0%|          | 0/3000 [00:00<?, ? examples/s]

==((====))==  Unsloth - 2x faster free finetuning | Num GPUs used = 1
   \\   /|    Num examples = 3,000 | Num Epochs = 3 | Total steps = 282
O^O/ \_/ \    Batch size per device = 32 | Gradient accumulation steps = 1
\        /    Data Parallel GPUs = 1 | Total batch size (32 x 1 x 1) = 32
 "-____-"     Trainable parameters = 67,502,080 of 3,916,592,128 (1.72% trained)


Unsloth: Will smartly offload gradients to save VRAM!


Step,Training Loss
10,1.903567
20,0.293817
30,0.188185
40,0.185942
50,0.168959
60,0.156389
70,0.147267
80,0.151989
90,0.140922
100,0.127808



LoRA training complete!


## 5. Test the Trained Model

The LoRA backend returns the trained model and tokenizer directly in the result dict, so we don't need to reload from a checkpoint.

In [8]:
from unsloth import FastLanguageModel

model = result["model"]
tokenizer = result["tokenizer"]

# Switch to inference mode for faster generation
FastLanguageModel.for_inference(model)


def generate_answer(question, max_tokens=256):
    messages = [{"role": "user", "content": question}]
    input_text = tokenizer.apply_chat_template(messages, tokenize=False, add_generation_prompt=True)
    # Use text= keyword to bypass Unsloth's VLM processor patching
    inputs = tokenizer(text=input_text, return_tensors="pt").to(model.device)
    input_len = inputs["input_ids"].shape[1]
    with torch.no_grad():
        outputs = model.generate(
            **inputs,
            max_new_tokens=max_tokens,
            do_sample=True,
            temperature=0.1,
            top_p=0.95,
            pad_token_id=tokenizer.eos_token_id,
        )
    return tokenizer.decode(outputs[0][input_len:], skip_special_tokens=True).strip()

In [9]:
test_questions = [
    "What are the classic signs and symptoms of diabetic ketoacidosis?",
    "What is the mechanism of action of ACE inhibitors in treating hypertension?",
    "What is the most common cause of iron deficiency anemia in adults?",
    "What are the differences between Type 1 and Type 2 diabetes mellitus?",
    "What is the triad of symptoms seen in nephrotic syndrome?",
]

print("=" * 70)
print("  LoRA-Trained Medical Q&A")
print("=" * 70)
for i, q in enumerate(test_questions):
    answer = generate_answer(q)
    print(f"\nQ{i+1}: {q}")
    print(f"A{i+1}: {answer}")
    print("-" * 70)

Both `max_new_tokens` (=256) and `max_length`(=262144) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)


  LoRA-Trained Medical Q&A


Both `max_new_tokens` (=256) and `max_length`(=262144) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)



Q1: What are the classic signs and symptoms of diabetic ketoacidosis?
A1: The classic signs and symptoms of diabetic ketoacidosis include polyuria, polydipsia, and fruity-smelling breath.
----------------------------------------------------------------------


Both `max_new_tokens` (=256) and `max_length`(=262144) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)



Q2: What is the mechanism of action of ACE inhibitors in treating hypertension?
A2: ACE inhibitors work by inhibiting the conversion of angiotensin I to angiotensin II, which leads to decreased vasoconstriction and decreased aldosterone secretion.
----------------------------------------------------------------------


Both `max_new_tokens` (=256) and `max_length`(=262144) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)



Q3: What is the most common cause of iron deficiency anemia in adults?
A3: The most common cause of iron deficiency anemia in adults is blood loss, which can occur due to a variety of reasons such as heavy menstrual bleeding, gastrointestinal bleeding, or injury. Iron deficiency anemia is a condition in which the body does not have enough iron to produce hemoglobin, which is the protein in red blood cells that carries oxygen throughout the body. Without enough iron, the body cannot produce enough hemoglobin, leading to a decrease in the number of red blood cells and a decrease in the ability to carry oxygen. Symptoms of iron deficiency anemia can include fatigue, weakness, pale skin, shortness of breath, and headaches. Treatment for iron deficiency anemia typically involves iron supplementation, either through dietary changes or supplements, and addressing the underlying cause of blood loss.
----------------------------------------------------------------------


Both `max_new_tokens` (=256) and `max_length`(=262144) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)



Q4: What are the differences between Type 1 and Type 2 diabetes mellitus?
A4: Type 1 diabetes mellitus is an autoimmune disorder that results in absolute insulin deficiency, while Type 2 diabetes mellitus is a disorder of insulin resistance with relative insulin deficiency.
----------------------------------------------------------------------

Q5: What is the triad of symptoms seen in nephrotic syndrome?
A5: The triad of symptoms seen in nephrotic syndrome is hypoalbuminemia, edema, and hypercoagulability.
----------------------------------------------------------------------


## 6. Save and Load the Model (Optional)

The model was already saved during training. You can reload it later:

In [10]:
# List saved files
ckpt_files = list(Path(CKPT_DIR).glob("*"))
print(f"Checkpoint directory: {CKPT_DIR}")
for f in sorted(ckpt_files):
    size_mb = f.stat().st_size / 1e6 if f.is_file() else 0
    print(f"  {f.name}" + (f"  ({size_mb:.1f} MB)" if size_mb > 0 else "/"))

Checkpoint directory: ./ministral_lora_output/checkpoints
  README.md  (0.0 MB)
  adapter_config.json  (0.0 MB)
  adapter_model.safetensors  (270.1 MB)
  chat_template.jinja  (0.0 MB)
  checkpoint-282/
  processor_config.json  (0.0 MB)
  tokenizer.json  (17.1 MB)
  tokenizer_config.json  (0.0 MB)
  training_args.bin  (0.0 MB)
  training_metrics.jsonl  (0.0 MB)


In [11]:
# To reload the model later:
#
# from unsloth import FastLanguageModel
# model, tokenizer = FastLanguageModel.from_pretrained(CKPT_DIR)
# FastLanguageModel.for_inference(model)

## Summary

In this notebook we:

1. **Loaded** the medical flashcards dataset and converted it to JSONL messages format
2. **Fine-tuned** Ministral 3 3B using LoRA on 3,000 medical Q&A pairs
3. **Verified** the trained model produces factual medical answers

**Key takeaways:**
- LoRA trains only ~1–2% of model parameters, making it feasible on a single GPU
- QLoRA (4-bit) reduces memory further — set `USE_QLORA = True` for GPUs with <40GB VRAM
- The Unsloth backend provides optimized LoRA training with automatic patching

**Next steps:**
- Compare with SFT (`runnable_ministral_sft.ipynb`) and OSFT (`runnable_ministral_osft.ipynb`)
- Scale up to the full 33,955-sample dataset
- Experiment with rank (`lora_r`) — higher ranks give more capacity but use more memory
- Use `plot_loss()` from Training Hub to visualize the training curve